In [1]:
import numpy as np
import scipy.sparse as sp
import faiss
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

import sys
from pathlib import Path
root = Path.cwd()
if not (root / "modules").is_dir():
    REPO_ROOT = root.parent
sys.path.insert(0, str(REPO_ROOT))
from utilities.data_loader import load_embeddings
from utilities.utils import *

In [2]:
def target_distribution(q):
    weight = (q ** 2) / torch.sum(q, dim=0)
    return (weight.t() / torch.sum(weight, dim=1)).t()


def cluster_delta(y_prev, y_curr):
    cm = confusion_matrix(y_prev, y_curr)
    row_ind, col_ind = linear_sum_assignment(-cm)
    matched = cm[row_ind, col_ind].sum()
    return 1 - matched / len(y_prev)


def build_knn_graph(
    X,
    topk=10,
    metric="cosine"
):
    """
    Unified graph construction for BOTH:
    - TF-IDF + PCA/SVD
    - SBERT embeddings

    Uses cosine similarity by default.
    """

    X = X.astype(np.float32).copy()

    N, d = X.shape

    if metric == "cosine":

        faiss.normalize_L2(X)

        index = faiss.IndexFlatIP(d)

    elif metric == "euclidean":

        index = faiss.IndexFlatL2(d)

    else:
        raise ValueError(f"Unsupported metric: {metric}")

    index.add(X)

    sims, indices = index.search(X, topk + 1)

    rows = []
    cols = []
    vals = []

    for i in range(N):

        for j, sim in zip(indices[i][1:], sims[i][1:]):

            rows.append(i)
            cols.append(j)

            if metric == "euclidean":
                sim = np.exp(-sim)

            vals.append(float(sim))

    adj = sp.coo_matrix(
        (vals, (rows, cols)),
        shape=(N, N),
        dtype=np.float32
    )

    # Symmetrize
    adj = adj.maximum(adj.T)

    # Self-loops
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32)

    deg = np.array(adj.sum(1)).flatten()

    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-10)

    D_inv_sqrt = sp.diags(deg_inv_sqrt)

    adj = D_inv_sqrt @ adj @ D_inv_sqrt

    adj = adj.tocsr()
    adj.eliminate_zeros()

    return adj


def sparse_to_torch(adj):

    adj = adj.tocoo()

    indices = torch.from_numpy(
        np.vstack((adj.row, adj.col))
    ).long()

    values = torch.from_numpy(adj.data).float()

    shape = torch.Size(adj.shape)

    return torch.sparse_coo_tensor(
        indices,
        values,
        shape
    )


class AE(nn.Module):

    def __init__(
        self,
        n_input,
        n_enc_1=500,
        n_enc_2=500,
        n_enc_3=2000,
        n_z=128,
        dropout=0.2
    ):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        # Encoder
        self.enc_1 = nn.Linear(n_input, n_enc_1)
        self.enc_2 = nn.Linear(n_enc_1, n_enc_2)
        self.enc_3 = nn.Linear(n_enc_2, n_enc_3)
        self.z_layer = nn.Linear(n_enc_3, n_z)

        # Decoder
        self.dec_1 = nn.Linear(n_z, n_enc_3)
        self.dec_2 = nn.Linear(n_enc_3, n_enc_2)
        self.dec_3 = nn.Linear(n_enc_2, n_enc_1)
        self.x_bar = nn.Linear(n_enc_1, n_input)

    def forward(self, x):

        h1 = self.dropout(F.relu(self.enc_1(x)))
        h2 = self.dropout(F.relu(self.enc_2(h1)))
        h3 = self.dropout(F.relu(self.enc_3(h2)))

        z = self.z_layer(h3)

        d1 = F.relu(self.dec_1(z))
        d2 = F.relu(self.dec_2(d1))
        d3 = F.relu(self.dec_3(d2))

        x_hat = self.x_bar(d3)

        return x_hat, h1, h2, h3, z


class GCNLayer(nn.Module):

    def __init__(self, in_dim, out_dim):

        super().__init__()

        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj, active=True):

        x = self.linear(x)

        x = torch.sparse.mm(adj, x)

        if active:
            x = F.relu(x)

        return x


class SDCN(nn.Module):

    def __init__(
        self,
        n_input,
        n_z,
        n_clusters,
        sigma=0.5
    ):

        super().__init__()

        self.sigma = sigma

        # AE
        self.ae = AE(
            n_input=n_input,
            n_enc_1=500,
            n_enc_2=500,
            n_enc_3=2000,
            n_z=n_z
        )

        # GCN
        self.gnn_1 = GCNLayer(n_input, 500)
        self.gnn_2 = GCNLayer(500, 500)
        self.gnn_3 = GCNLayer(500, 2000)
        self.gnn_4 = GCNLayer(2000, n_z)
        self.gnn_5 = GCNLayer(n_z, n_clusters)

        # Clustering layer
        self.cluster_layer = nn.Parameter(
            torch.Tensor(n_clusters, n_z)
        )

        nn.init.xavier_normal_(self.cluster_layer.data)

        self.v = 1.0

    def soft_assign(self, z):

        dist = torch.sum(
            (z.unsqueeze(1) - self.cluster_layer) ** 2,
            dim=2
        )

        q = 1.0 / (1.0 + dist / self.v)
        q = q ** ((self.v + 1.0) / 2.0)
        return q / q.sum(dim=1, keepdim=True)

    def forward(self, x, adj):

        # AE
        x_bar, h1, h2, h3, z = self.ae(x)

        s = self.sigma
        # GCN + fusion

        g1 = self.gnn_1(x, adj)
        g2 = self.gnn_2(
            (1 - s) * g1 + s * h1,
            adj
        )
        g3 = self.gnn_3(
            (1 - s) * g2 + s * h2,
            adj
        )
        g4 = self.gnn_4(
            (1 - s) * g3 + s * h3,
            adj
        )

        g5 = self.gnn_5(
            (1 - s) * g4 + s * z,
            adj,
            active=False
        )

        pred = F.log_softmax(g5, dim=1)

        q = self.soft_assign(z)

        return x_bar, q, pred, z


class SDCNClusterer:

    def __init__(
        self,
        device="cuda",
        latent_dim=128,
        knn_k=10,
        sigma=0.5,
        alpha=0.1,
        beta=1.0,
        lr=1e-3,
        pretrain_epochs=100,
        epochs=200,
        update_interval=10,
        tol=1e-3,
        use_pca=False
    ):

        self.device = device
        self.latent_dim = latent_dim

        self.knn_k = knn_k

        self.sigma = sigma

        self.alpha = alpha
        self.beta = beta

        self.lr = lr

        self.pretrain_epochs = pretrain_epochs
        self.epochs = epochs

        self.update_interval = update_interval
        self.tol = tol

        self.use_pca = use_pca

        self.training_history = []

    def fit_predict(
        self,
        embeddings,
        k,
        true_labels=None
    ):

        torch.manual_seed(42)
        np.random.seed(42)

        self.training_history = []

        best_metrics = {
            "epoch": -1,
            "NMI": -1,
            "ARI": -1,
            "ACC": -1,
            "Purity": -1
        }

        device = torch.device(self.device)

        # Optional PCA

        if self.use_pca:

            X_np = PCA(
                n_components=0.95,
                random_state=42
            ).fit_transform(embeddings)

        else:
            X_np = embeddings

        # Build graph

        adj = build_knn_graph(
            X_np,
            topk=self.knn_k,
            metric="cosine"
        )

        # Diagnostics

        density = adj.nnz / (adj.shape[0] ** 2)

        print(f"[Graph] Nodes: {adj.shape[0]}")
        print(f"[Graph] Edges: {adj.nnz}")
        print(f"[Graph] Density: {density:.8f}")

        self.training_history.append({
            "stage": "graph",
            "nodes": int(adj.shape[0]),
            "edges": int(adj.nnz),
            "density": float(density),
            "knn_k": int(self.knn_k)
        })

        # Torch tensors

        X = torch.tensor(
            X_np,
            dtype=torch.float32,
            device=device
        )

        adj = sparse_to_torch(adj).coalesce().to(device)

        # Model

        model = SDCN(
            n_input=X.shape[1],
            n_z=self.latent_dim,
            n_clusters=k,
            sigma=self.sigma
        ).to(device)

        # AE pretraining

        optimizer_ae = torch.optim.Adam(
            model.ae.parameters(),
            lr=self.lr
        )

        print("\n[SDCN] Pretraining AE...\n")

        model.train()

        for epoch in range(self.pretrain_epochs):

            x_hat, *_ = model.ae(X)

            loss = F.mse_loss(x_hat, X)

            optimizer_ae.zero_grad()

            loss.backward()

            optimizer_ae.step()

            self.training_history.append({
                "stage": "pretrain",
                "epoch": int(epoch),
                "reconstruction_loss": float(loss.item())
            })

        # KMeans init

        with torch.no_grad():

            _, _, _, _, z = model.ae(X)

        z_np = z.detach().cpu().numpy()

        kmeans = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=42
        )

        y_pred = kmeans.fit_predict(z_np)

        y_pred_last = y_pred.copy()

        model.cluster_layer.data = torch.tensor(
            kmeans.cluster_centers_,
            dtype=torch.float32,
            device=device
        )

        # Joint training

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=self.lr
        )

        print("\n[SDCN] Joint Training...\n")

        model.train()

        for epoch in range(self.epochs):

            x_bar, q, pred, z = model(X, adj)

            q = q.clamp(min=1e-10)

            p = target_distribution(q).detach()

            # Losses
            kl_loss = F.kl_div(
                q.log(),
                p,
                reduction="batchmean"
            )

            ce_loss = F.kl_div(
                pred,
                p,
                reduction="batchmean",
                log_target=False
            )

            re_loss = F.mse_loss(x_bar, X)

            loss = (
                self.alpha * kl_loss +
                self.beta * ce_loss +
                re_loss
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            # ---------------------------------------------
            # Convergence
            # ---------------------------------------------
            delta = None

            if epoch % self.update_interval == 0:

                with torch.no_grad():

                    y_pred = q.argmax(1).cpu().numpy()

                    delta = cluster_delta(
                        y_pred_last,
                        y_pred
                    )

                    y_pred_last = y_pred.copy()

                if true_labels is not None:

                    metrics = get_metrics(
                        X_np,
                        true_labels,
                        y_pred
                    )

                    self.training_history.append({
                        "stage": "eval",
                        "epoch": int(epoch),
                        "NMI": float(metrics["NMI"]),
                        "ARI": float(metrics["ARI"]),
                        "ACC": float(metrics["Accuracy"]),
                        "Purity": float(metrics["Purity"])
                    })

                    # Track best metrics
                    if metrics["Accuracy"] > best_metrics["ACC"]:

                        best_metrics = {
                            "epoch": int(epoch),
                            "NMI": float(metrics["NMI"]),
                            "ARI": float(metrics["ARI"]),
                            "ACC": float(metrics["Accuracy"]),
                            "Purity": float(metrics["Purity"])
                        }

                if epoch > 20 and delta < self.tol:

                    print(f"[SDCN] Converged at epoch {epoch}")

                    self.training_history.append({
                        "stage": "cluster",
                        "epoch": int(epoch),
                        "loss": float(loss.item()),
                        "kl_loss": float(kl_loss.item()),
                        "ce_loss": float(ce_loss.item()),
                        "reconstruction_loss": float(re_loss.item()),
                        "delta": float(delta),
                        "converged": True
                    })

                    break

            # ---------------------------------------------
            # Logging
            # ---------------------------------------------
            self.training_history.append({
                "stage": "cluster",
                "epoch": int(epoch),
                "loss": float(loss.item()),
                "kl_loss": float(kl_loss.item()),
                "ce_loss": float(ce_loss.item()),
                "reconstruction_loss": float(re_loss.item()),
                "delta": None if delta is None else float(delta)
            })

            if epoch % 10 == 0:

                print(
                    f"[SDCN] Epoch {epoch} | "
                    f"Loss={loss.item():.4f} | "
                    f"KL={kl_loss.item():.4f} | "
                    f"CE={ce_loss.item():.4f} | "
                    f"RE={re_loss.item():.4f}"
                )

        # Final prediction
        model.eval()
        with torch.no_grad():

            _, q, _, _ = model(X, adj)

            labels = q.argmax(1).cpu().numpy()

        if true_labels is not None:
            final_metrics = get_metrics(
                X_np,
                true_labels,
                labels
            )

            self.training_history.append({
                "stage": "final",
                "NMI": float(final_metrics["NMI"]),
                "ARI": float(final_metrics["ARI"]),
                "ACC": float(final_metrics["Accuracy"]),
                "Purity": float(final_metrics["Purity"])
            })

            self.training_history.append({
                "stage": "best",
                **best_metrics
            })

        return labels, self.training_history

In [3]:
def run_sdcn(
    X,
    n_clusters,
    device,
    y_true=None,
    dataset_name="unknown"
):

    model = SDCNClusterer(
        device=device,
        latent_dim=128,
        knn_k=10,
        sigma=0.5,
        alpha=0.1,
        beta=1.0,
        lr=1e-3,
        pretrain_epochs=100,
        epochs=200,
        update_interval=10,
        tol=1e-3,
        use_pca=False
    )

    y_pred, history = model.fit_predict(
        embeddings=X,
        k=n_clusters,
        true_labels=y_true
    )

    return y_pred, history


def run_experiments(datasets_root, csv_path):

    datasets = list(Path(datasets_root).glob("*.npz"))

    history_dict = {}

    for dataset_path in datasets:

        encoder_name = dataset_path.stem.split("_")[2]

        dataset_name = dataset_path.stem.replace(
            "emb_", ""
        ).replace(f"_{encoder_name}", "")

        print(
            f"\nRunning dataset: {dataset_name} "
            f"with encoder: {encoder_name}"
        )

        X, y, texts = load_embeddings(dataset_path)

        X_reduced, pca = pca_by_variance(X)

        pca_components = pca.n_components_

        k_true = len(set(y))

        y_pred, history = run_sdcn(
            X=X_reduced,
            n_clusters=k_true,
            device=get_device(),
            y_true=y,
            dataset_name=dataset_name
        )

        history_dict[
            dataset_name + "_" + encoder_name
        ] = history

        metrics = get_metrics(
            X_reduced,
            y,
            y_pred
        )

        plot_and_save_clusters(
            dataset=dataset_name,
            encoder_name=encoder_name,
            clusterer="SDCN",
            X=X_reduced,
            y_pred=y_pred,
            n_clusters=k_true,
            number_of_components=pca_components
        )

        log_experiment(
            csv_path=csv_path,
            model_name="SDCN",
            dataset_name=dataset_name,
            encoder_name=encoder_name,
            n_rows=len(X_reduced),
            pca_components=pca_components,
            nmi=metrics["NMI"],
            ari=metrics["ARI"],
            acc=metrics["Accuracy"],
            purity=metrics["Purity"]
        )

    return history_dict

In [ ]:
history = run_experiments(
        datasets_root="../embeddings",
        csv_path="../outputs/results.csv"
    )


Running dataset: agnews with encoder: tfidf
[Graph] Nodes: 127600
[Graph] Edges: 2171884
[Graph] Density: 0.00013339
